# Dreamers-AI — Empirical check of the fine-tuned LoRA adapter

Loads **`sgogoi/Llama-fine-tune-movies`** (LoRA/PEFT) on top of the gated base **`meta-llama/Llama-3.2-3B-Instruct`**, generates next-scene continuations from real validation prompts, and measures the three things in question:

1. **Format** — does it produce screenplay text / start with an `INT.`/`EXT.` slug?
2. **OCR contamination** — does it echo the form-feed (`\f`) page-break junk present in ~43% of training data?
3. **Stopping** — does it emit `<|end_of_scene|>` and stop cleanly? (It was trained as *plain text*, not a registered EOS token, so we stop on the literal string.)

**Prereqs:** a GPU runtime (Colab T4 is enough) and you must have **accepted the license** for `meta-llama/Llama-3.2-3B-Instruct` on its HF model page with the same account as your token.

> Prompt format note: the adapter was trained with axolotl `type: alpaca` and `train_on_inputs: false`, so inference uses the **Alpaca template**, not the Llama chat template. If outputs look off, flip `USE_ALPACA = False` in the config cell to try the chat template.

## 1. Install dependencies

In [1]:
# Colab already ships a CUDA build of torch; we add the rest.
%pip install -q -U transformers peft accelerate bitsandbytes datasets huggingface_hub

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\u26a0\ufe0f  No GPU detected — set Runtime > Change runtime type > GPU.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.5 MB/s eta 0:00:00
torch 2.11.0+cu128 | CUDA available: True
GPU: Tesla T4


## 2. Hugging Face login (needed for the gated base model)

In [2]:
from huggingface_hub import login
# Paste a token with read access (https://huggingface.co/settings/tokens).
# Make sure that account has accepted the Llama-3.2-3B-Instruct license.
login()

## 3. Configuration

In [3]:
BASE_MODEL   = 'meta-llama/Llama-3.2-3B-Instruct'
ADAPTER      = 'sgogoi/Llama-fine-tune-movies'
DATASET_REPO = 'sgogoi/movie-scripts'   # contains val.jsonl
VAL_FILE     = 'val.jsonl'

STOP_STR     = '<|end_of_scene|>'
N_PROMPTS    = 6        # how many val examples to generate from
MAX_NEW      = 400      # max new tokens per generation
USE_ALPACA   = True     # True = alpaca template (matches training); False = Llama chat template
LOAD_4BIT    = True     # QLoRA-style 4-bit load (matches training; fits a T4 easily)
SEED         = 42

## 4. Load base model + LoRA adapter

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

quant = None
if LOAD_4BIT:
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()
print('\u2705 Base + adapter loaded.')

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/389M [00:00<?, ?B/s]

✅ Base + adapter loaded.


## 5. Fetch real validation prompts

In [5]:
import json
from huggingface_hub import hf_hub_download

path = hf_hub_download(repo_id=DATASET_REPO, filename=VAL_FILE, repo_type='dataset')
examples = []
with open(path, encoding='utf-8') as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        examples.append(json.loads(line))
        if len(examples) >= N_PROMPTS:
            break

print(f'Loaded {len(examples)} validation examples.')
print('Keys:', list(examples[0].keys()))
print('\nFirst input (truncated):\n', examples[0]['input'][:400])

val.jsonl:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

Loaded 6 validation examples.
Keys: ['instruction', 'input', 'output', 'source_file']

First input (truncated):
 {"movie_details": {"genre": "Psychological Thriller", "theme": "Survival, paranoia, and the psychological struggle against an uncertain, escalating threat.", "tone": "Tense, claustrophobic, and suspenseful"}, "previous_scene": "INT. STAIRWELL - EVENING\n\nMichelle arrives at the top of the stairs and finds a thick\nlead door with a small 4x6 inch window.\n\nHoward comes halfway up the stairs behin


## 6. Prompt builder + string-based stopping criteria

In [6]:
from transformers import StoppingCriteria, StoppingCriteriaList

ALPACA_TMPL = (
    'Below is an instruction that describes a task, paired with an input that '
    'provides further context. Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n'
)

def build_prompt(ex):
    if USE_ALPACA:
        return ALPACA_TMPL.format(instruction=ex.get('instruction', ''), input=ex.get('input', ''))
    # Llama chat template fallback
    msg = [{'role': 'user', 'content': ex.get('instruction', '') + '\n\n' + ex.get('input', '')}]
    return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

class StopOnString(StoppingCriteria):
    """Stop when the decoded continuation contains the literal stop string."""
    def __init__(self, stop_str, tokenizer, prompt_len):
        self.stop_str, self.tok, self.prompt_len = stop_str, tokenizer, prompt_len
    def __call__(self, input_ids, scores, **kw):
        text = self.tok.decode(input_ids[0][self.prompt_len:], skip_special_tokens=True)
        return self.stop_str in text

## 7. Generate

In [7]:
import torch
torch.manual_seed(SEED)

results = []
for i, ex in enumerate(examples):
    prompt = build_prompt(ex)
    enc = tok(prompt, return_tensors='pt').to(model.device)
    plen = enc['input_ids'].shape[1]
    stopper = StoppingCriteriaList([StopOnString(STOP_STR, tok, plen)])
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tok.pad_token_id,
            stopping_criteria=stopper,
        )
    gen = tok.decode(out[0][plen:], skip_special_tokens=True)
    results.append({'input': ex.get('input', ''), 'reference': ex.get('output', ''), 'generated': gen})

    print('=' * 100)
    print(f'EXAMPLE {i + 1}/{len(examples)}')
    print('-- GENERATED --')
    print(gen[:1200])
    print('\n-- REFERENCE (truncated) --')
    print(ex.get('output', '')[:400])
    print()

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


EXAMPLE 1/6
-- GENERATED --
EXT. BACKYARD - EVENING

Howard guides Michelle to a rusty white pickup truck with
torn-out windows.

HOWARD
You seen this? It’s your ride.
Get inside. Close the doors,
and roll up all the windows.

The tailgate pops open. Howard pushes Michelle into the back
seat and shuts the door.

HOWARD (CONT’D)
Just drive it around for awhile
if you have to. Keep it going.

He puts his hand on the driver's side door and gives it a
little push --

HOWARD (CONT’D)
Here we go.

And he pushes it again, harder this time, and it swings shut.
<|end_of_scene|>

-- REFERENCE (truncated) --
INT. TOP OF STAIRS - EVENING
Michelle pulls away. Terror pulses through her body.

HOWARD
Believe me now?

Howard climbs the stairs as Michelle tries to get her
panicked breathing under control.

HOWARD (CONT'D)
Mildred and Frank. They seemed fine
when we got here. I looked out a
few hours later. That’s what I saw.

MICHELLE
What happened to them?

HOWARD
I don’t know.

Howard flips a switch 

## 8. Automated quality analysis

In [8]:
import re
FF = chr(12)  # form-feed, the OCR page-break artifact
SLUG = re.compile(r'\b(INT|EXT)[\.\s/]', re.I)
n = len(results)

emit_stop  = sum(1 for r in results if STOP_STR in r['generated'])
starts_slug = sum(1 for r in results if SLUG.search(r['generated'][:60]))
has_slug    = sum(1 for r in results if SLUG.search(r['generated']))
has_ff      = sum(1 for r in results if FF in r['generated'])
# 'runaway' = used the whole budget without ever emitting the stop string
runaway     = sum(1 for r in results if STOP_STR not in r['generated'] and len(r['generated']) > 200)

def pc(x):
    return f'{100.0 * x / n:5.1f}%  ({x}/{n})'

print('GENERATION QUALITY (n =', n, ')\n')
print('emits <|end_of_scene|> & stops:  ', pc(emit_stop))
print('starts with an INT./EXT. slug:   ', pc(starts_slug))
print('contains any INT./EXT. slug:     ', pc(has_slug))
print('echoes form-feed OCR artifact:   ', pc(has_ff), '  <- contamination learned from data')
print('runaway (no stop, hit budget):   ', pc(runaway))

avg_len = sum(len(r['generated']) for r in results) / n
print(f'\navg generated length: {avg_len:.0f} chars')

GENERATION QUALITY (n = 6 )

emits <|end_of_scene|> & stops:   100.0%  (6/6)
starts with an INT./EXT. slug:     83.3%  (5/6)
contains any INT./EXT. slug:       83.3%  (5/6)
echoes form-feed OCR artifact:     33.3%  (2/6)   <- contamination learned from data
runaway (no stop, hit budget):      0.0%  (0/6)

avg generated length: 367 chars


## How to read the results

| Signal | Good | Bad (and what it means) |
|---|---|---|
| **emits `<|end_of_scene|>`** | high | low → model didn't learn to terminate; generation runs to the token budget |
| **starts with slug** | high-ish | low is *expected* — ~40% of training outputs were paragraph chunks, not full slugged scenes |
| **form-feed artifact** | ~0% | any non-trivial % confirms the model echoes OCR junk baked into the data |
| **runaway** | ~0% | high → stop-string learning failed; needs a hard `max_new_tokens` + string-stop at serve time |

If outputs are coherent screenplay text that stop cleanly, the adapter is usable despite the dirty data. If they're fragmentary, echo `\f`, or never stop, that's the data problem showing through — fix = clean the corpus (strip `\f`/boilerplate) and retrain.